In [1]:
import pandas as pd


In [2]:
path = "online_retail_II.xlsx"
df1 = pd.read_excel(path, sheet_name="Year 2009-2010")
df2 = pd.read_excel(path, sheet_name="Year 2010-2011")

df = pd.concat([df1, df2], ignore_index=True)
df.to_csv("online_retail_II.csv", index=False)
print(df.shape)

(1067371, 8)


In [3]:
df =pd.read_csv("online_retail_II.csv",parse_dates=["InvoiceDate"])

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 65.1 MB


In [5]:
print(df.isnull().sum())

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [7]:
print(df.describe())

           Quantity                 InvoiceDate         Price    Customer ID
count  1.067371e+06                     1067371  1.067371e+06  824364.000000
mean   9.938898e+00  2011-01-02 21:13:55.394029  4.649388e+00   15324.638504
min   -8.099500e+04         2009-12-01 07:45:00 -5.359436e+04   12346.000000
25%    1.000000e+00         2010-07-09 09:46:00  1.250000e+00   13975.000000
50%    3.000000e+00         2010-12-07 15:28:00  2.100000e+00   15255.000000
75%    1.000000e+01         2011-07-22 10:23:00  4.150000e+00   16797.000000
max    8.099500e+04         2011-12-09 12:50:00  3.897000e+04   18287.000000
std    1.727058e+02                         NaN  1.235531e+02    1697.464450


In [9]:
raw_rows = len(df)

In [10]:
# 1. Cancelled orders: Invoice 'C' se start hota hai
cancelled = df["Invoice"].astype(str).str.startswith("C")
print("Cancelled:", cancelled.sum())

Cancelled: 19494


In [11]:
print("Missing Customer ID:", df["Customer ID"].isnull().sum())

Missing Customer ID: 243007


In [12]:
print("Bad qty/price:", ((df["Quantity"] <= 0) | (df["Price"] <= 0)).sum())


Bad qty/price: 25700


In [13]:
print("Duplicates:", df.duplicated().sum())


Duplicates: 34335


In [14]:
# Clean version
clean = df[~cancelled].copy()

In [15]:
clean = clean.dropna(subset=["Customer ID"])

In [16]:
clean = clean[(clean["Quantity"] > 0) & (clean["Price"] > 0)]

In [17]:
clean = clean.drop_duplicates()

In [18]:
clean["Customer ID"] = clean["Customer ID"].astype(int)

In [19]:
clean["Revenue"] = clean["Quantity"] * clean["Price"]

In [20]:
print(raw_rows, "->", len(clean))

1067371 -> 779425


In [21]:
clean.to_csv("clean_retail.csv", index=False)

In [22]:
print("Total revenue:", clean["Revenue"].sum())

Total revenue: 17374804.268


In [23]:
print("Customers:", clean["Customer ID"].nunique())

Customers: 5878


In [24]:
print("Orders:", clean["Invoice"].nunique())

Orders: 36969


In [25]:
print("AOV:", clean.groupby("Invoice")["Revenue"].sum().mean())

AOV: 469.98307414320107


In [26]:
print(clean.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(5))

Country
United Kingdom    1.438923e+07
EIRE              6.165705e+05
Netherlands       5.540381e+05
Germany           4.250197e+05
France            3.487690e+05
Name: Revenue, dtype: float64


In [27]:
import pandas as pd

In [28]:
df = pd.read_csv("clean_retail.csv", parse_dates=["InvoiceDate"])

In [29]:
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)


In [30]:
rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

In [31]:
# 1-5 score (5 = best)
rfm["R"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5]).astype(int)


In [32]:
def segment(r):
    if r["R"] >= 4 and r["F"] >= 4: return "Champions"
    if r["R"] >= 3 and r["F"] >= 3: return "Loyal"
    if r["R"] >= 4 and r["F"] <= 2: return "New/Promising"
    if r["R"] <= 2 and r["F"] >= 4: return "At Risk (High Value)"
    if r["R"] <= 2: return "Churned/Lost"
    return "Needs Attention"

In [33]:
rfm["Segment"] = rfm.apply(segment, axis=1)
rfm.to_csv("rfm_segments.csv", index=False)


In [34]:

summary = rfm.groupby("Segment").agg(
    Customers=("Customer ID", "count"),
    Revenue=("Monetary", "sum")
)

In [35]:
summary["Customer_%"] = (summary["Customers"] / summary["Customers"].sum() * 100).round(1)
summary["Revenue_%"] = (summary["Revenue"] / summary["Revenue"].sum() * 100).round(1)
print(summary.sort_values("Revenue", ascending=False))

                      Customers       Revenue  Customer_%  Revenue_%
Segment                                                             
Champions                  1482  1.202433e+07        25.2       69.2
Loyal                      1221  2.510046e+06        20.8       14.4
Churned/Lost               1994  1.153117e+06        33.9        6.6
At Risk (High Value)        353  1.090694e+06         6.0        6.3
New/Promising               443  3.922670e+05         7.5        2.3
Needs Attention             385  2.043497e+05         6.5        1.2
